# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [1]:
%uv pip install transformers datasets torch accelerate scikit-learn

Using Python 3.12.6 environment at: /usr/local
Resolved 68 packages in 493ms
⠙ Preparing packages... (0/5)
⠙ Preparing packages... (0/5)
dill       ------------------------------     0 B/117.21 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------     0 B/117.21 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 14.84 KiB/117.21 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 14.84 KiB/117.21 KiB
multiprocess ------------------------------     0 B/146.76 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 14.84 KiB/117.21 KiB
multiprocess ------------------------------     0 B/146.76 KiB
xxhash     ------------------------------ 14.87 KiB/189.39 KiB
⠙ Preparing packages... (0/5)
dill       ------------------------------ 14.84 KiB/117.21 KiB
multiprocess ------------------------------     0 B/146.76 KiB
xxhash     ------------------------------ 14.87 KiB/189.39 KiB
datasets   -------------

In [2]:
from datasets import load_dataset

print("Loading dataset...")

dataset = load_dataset("amazon_polarity")

print(dataset)

Loading dataset...


README.md: 0.00B [00:00, ?B/s]

amazon_polarity/train-00000-of-00004.par(…):   0%|          | 0.00/260M [00:00<?, ?B/s]

amazon_polarity/train-00001-of-00004.par(…):   0%|          | 0.00/258M [00:00<?, ?B/s]

amazon_polarity/train-00002-of-00004.par(…):   0%|          | 0.00/255M [00:00<?, ?B/s]

amazon_polarity/train-00003-of-00004.par(…):   0%|          | 0.00/254M [00:00<?, ?B/s]

amazon_polarity/test-00000-of-00001.parq(…):   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 3600000
    })
    test: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 400000
    })
})


In [3]:
train_data = dataset["train"].shuffle(seed=42).select(range(500000))
test_data = dataset["test"].shuffle(seed=42).select(range(20000))

print("Train samples:", len(train_data))
print("Test samples:", len(test_data))

Train samples: 500000
Test samples: 20000


In [4]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

print("Tokenizer loaded!")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Tokenizer loaded!


In [5]:
def tokenize(batch):
    return tokenizer(
        batch["content"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

print("Tokenizing training data...")

train_data = train_data.map(
    tokenize,
    batched=True,
    batch_size=1000
)

print("Tokenizing test data...")

test_data = test_data.map(
    tokenize,
    batched=True,
    batch_size=1000
)

print("Tokenization completed!")

Tokenizing training data...


Map:   0%|          | 0/500000 [00:00<?, ? examples/s]

Tokenizing test data...


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenization completed!


In [6]:
train_data.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_data.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

print("PyTorch format set!")

PyTorch format set!


In [7]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

print("Model loaded!")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded!


In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=1000,
    fp16=True
)

print("Training arguments ready!")

Training arguments ready!


In [9]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

print("Metrics ready!")

Metrics ready!


In [10]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics
)

print("Trainer ready!")

Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Trainer ready!


In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.150300,0.158045,0.943250,0.942095
2,0.092400,0.151245,0.949550,0.949673


TrainOutput(global_step=15626, training_loss=0.13525413328625094, metrics={'train_runtime': 551.3528, 'train_samples_per_second': 1813.721, 'train_steps_per_second': 28.341, 'total_flos': 3.3116849664e+16, 'train_loss': 0.13525413328625094, 'epoch': 2.0})

In [12]:
predictions = trainer.predict(test_data)

preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

print("Prediction completed!")

Prediction completed!


In [13]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

print("Accuracy:", accuracy_score(labels, preds))
print("F1 Score:", f1_score(labels, preds))

print(
    classification_report(
        labels,
        preds,
        target_names=["Negative", "Positive"]
    )
)

Accuracy: 0.94955
F1 Score: 0.9496733004139858
              precision    recall  f1-score   support

    Negative       0.95      0.95      0.95      9961
    Positive       0.95      0.95      0.95     10039

    accuracy                           0.95     20000
   macro avg       0.95      0.95      0.95     20000
weighted avg       0.95      0.95      0.95     20000



In [14]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

def predict_sentiment(review):

    inputs = tokenizer(
        review,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred = outputs.logits.argmax(-1).item()

    return "Positive 😊" if pred == 1 else "Negative 😞"

In [15]:
print(
    predict_sentiment(
        "This product is absolutely amazing!"
    )
)

print(
    predict_sentiment(
        "Worst purchase I have ever made."
    )
)

print(
    predict_sentiment(
        "The quality exceeded my expectations."
    )
)

print(
    predict_sentiment(
        "Completely useless and broke after one day."
    )
)

Positive 😊
Negative 😞
Positive 😊
Negative 😞


In [16]:
model.save_pretrained("./amazon_sentiment_model")
tokenizer.save_pretrained("./amazon_sentiment_model")

print("Model saved successfully!")

Model saved successfully!
